# Clasificación de Flores Iris

Proyecto de ciencia de datos listo para **Google Colab**.

**Pregunta:** ¿podemos predecir la especie de una flor Iris a partir de sus medidas?

**Target:** `species` (setosa, versicolor, virginica) — multi-clase.

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset clásico de Fisher (150 filas, 4 features, sin nulos).

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
df = sns.load_dataset("iris")
print(df.shape)
print(df["species"].value_counts())
df.head()

## 2. EDA

In [ ]:
df.describe().T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
for ax, col in zip(axes.flat, features):
    sns.histplot(data=df, x=col, hue="species", kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df, hue="species", diag_kind="kde", height=2.2)
plt.suptitle("Pairplot Iris", y=1.02)
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df.drop(columns="species").corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Correlación entre medidas")
plt.show()

### Hallazgos
- `setosa` se separa muy bien (pétalos pequeños).
- `versicolor` y `virginica` se solapan un poco en longitud de pétalo.
- `petal_length` y `petal_width` son las features más discriminativas.

## 3. Preparar datos

In [ ]:
X = df.drop(columns="species")
y = df["species"]

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.25, stratify=y_enc, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)
print("Clases:", le.classes_)

## 4. Modelos

In [ ]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "SVM": SVC(probability=True, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados.append({
        "modelo": nombre,
        "cv_acc_mean": scores.mean(),
        "cv_acc_std": scores.std(),
        "test_acc": accuracy_score(y_test, y_pred),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("test_acc", ascending=False)

## 5. Evaluación del mejor modelo

In [ ]:
mejor = "Random Forest"  # suele ganar o empatar
pipe = pipes[mejor]
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(cmap="Blues")
plt.title(f"Matriz de confusión — {mejor}")
plt.show()

## 6. Feature importance (Random Forest)

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(6, 4))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Importancia de features — Random Forest")
plt.xlabel("Importancia")
plt.tight_layout()
plt.show()
importancias

## 7. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_scaled = rf_pipe.named_steps["scaler"].transform(X_train)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled)

print("SHAP listo. Clases:", le.classes_)

In [ ]:
# Summary para una clase (virginica = índice 2)
shap.summary_plot(shap_values[2], X_train_scaled, feature_names=X.columns, show=False)
plt.title("SHAP Summary — clase virginica")
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values[2], X_train_scaled, feature_names=X.columns, plot_type="bar", show=False)
plt.title("Importancia media SHAP — virginica")
plt.tight_layout()
plt.show()

### Interpretación SHAP
- `petal_length` y `petal_width` dominan la separación de especies.
- `setosa` es casi linealmente separable; `versicolor` vs `virginica` depende más del tamaño del pétalo.

## 8. Predicción de ejemplo

In [ ]:
ejemplo = pd.DataFrame({
    "sepal_length": [5.1, 6.5],
    "sepal_width": [3.5, 3.0],
    "petal_length": [1.4, 5.2],
    "petal_width": [0.2, 2.0],
})

proba = pipes["Random Forest"].predict_proba(ejemplo)
pred = pipes["Random Forest"].predict(ejemplo)

pd.DataFrame({
    "perfil": ["pétalo pequeño", "pétalo grande"],
    "predicho": le.inverse_transform(pred),
    "prob_setosa": proba[:, 0].round(3),
    "prob_versicolor": proba[:, 1].round(3),
    "prob_virginica": proba[:, 2].round(3),
})

## 9. Conclusiones

- Dataset pequeño y limpio: accuracy de test suele estar en **0.95–1.00**.
- Las medidas del pétalo son las más importantes (EDA + importance + SHAP coinciden).
- `setosa` es fácil de separar; el error residual está entre versicolor y virginica.
- Ideal como segundo proyecto después del Titanic: menos limpieza, más foco en multi-clase y visualización.

**Cómo subirlo de nivel:**
- Visualizar fronteras de decisión en 2D (petal_length vs petal_width).
- Probar otros algoritmos (XGBoost, GaussianNB).
- Añadir SHAP dependence plots.